In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 1. REPRESENTATIVE STRUCTURE MATERIALS
# ==========================================
RIG_MATERIALS = {
    "Aluminum 6061": {"E": 68.9e9, "rho": 2700.0, "zeta_def": 0.015},
    "3D-Printed PLA": {"E": 2.8e9,  "rho": 1240.0, "zeta_def": 0.030},
    "Balsa Wood":     {"E": 3.5e9,  "rho": 160.0,  "zeta_def": 0.025},
}

STRUCTURE_TYPES = {
    "Cantilever Beam":        {"beta1": 1.8751, "beta2": 4.6941, "beta3": 7.8548, "eff_mass_ratio": 0.24},
    "Simply Supported Beam":  {"beta1": np.pi,  "beta2": 2*np.pi,"beta3": 3*np.pi,"eff_mass_ratio": 0.49},
    "Small Frame Structure":  {"beta1": 2.3500, "beta2": 5.1000, "beta3": 8.2000, "eff_mass_ratio": 0.32},
}

# ==========================================
# 2. NATURAL FREQUENCIES & FRF MODEL
# ==========================================
def compute_vibration_response(mat_name, struct_type, L_mm, b_mm, h_mm, tip_mass_g, zeta, f_max_hz, amp_g):
    mat = RIG_MATERIALS[mat_name]
    st = STRUCTURE_TYPES[struct_type]

    L = L_mm * 1e-3
    b = b_mm * 1e-3
    h = h_mm * 1e-3
    m_tip = tip_mass_g * 1e-3

    A = b * h
    I = (b * h**3) / 12.0
    m_beam = mat["rho"] * A * L
    m_eff = max(st["eff_mass_ratio"] * m_beam + m_tip, 1e-6)

    # First three natural frequencies [Hz] (mode-weighted mass effect for tip mass)
    f_list = []
    weights = [1.0, 0.35, 0.15]
    for idx, beta in enumerate([st["beta1"], st["beta2"], st["beta3"]]):
        m_mode = max(st["eff_mass_ratio"] * m_beam + weights[idx] * m_tip, 1e-6)
        fn = (beta**2 / (2.0 * np.pi * L**2)) * np.sqrt((mat["E"] * I) / (m_mode * L / max(st["eff_mass_ratio"], 0.1)))
        # Clean physical beam natural frequency formula: f_n = (beta_n^2 / (2*pi)) * sqrt(E*I / (m_total * L^3))
        m_total = max(m_beam + weights[idx] * m_tip, 1e-6)
        fn = (beta**2 / (2.0 * np.pi)) * np.sqrt((mat["E"] * I) / (m_total * (L**3)))
        f_list.append(fn)

    freqs = np.linspace(1.0, max(f_max_hz, 10.0), 1200)
    # Multi-mode dynamic magnification factor (displacement/acceleration response proxy)
    resp = np.zeros_like(freqs)
    for k_idx, fn in enumerate(f_list):
        if fn <= f_max_hz * 1.35:
            r = freqs / max(fn, 1e-6)
            mag = 1.0 / np.sqrt((1.0 - r**2)**2 + (2.0 * zeta * r)**2)
            mode_gain = 1.0 / ((k_idx + 1)**1.4)
            resp += amp_g * mode_gain * mag

    # Identify peaks inside sweep range [1.0, f_max_hz]
    peaks = []
    for idx, fn in enumerate(f_list):
        if 1.0 <= fn <= f_max_hz:
            r = fn / max(fn, 1e-6)
            mag = 1.0 / np.sqrt((1.0 - r**2)**2 + (2.0 * zeta * r)**2)
            # Total response around fn
            total_at_fn = 0.0
            for j_idx, f_other in enumerate(f_list):
                rj = fn / max(f_other, 1e-6)
                total_at_fn += amp_g * (1.0 / ((j_idx + 1)**1.4)) / np.sqrt((1.0 - rj**2)**2 + (2.0 * zeta * rj)**2)
            peaks.append({"Mode": f"Mode {idx+1}", "Freq_Hz": fn, "Response": total_at_fn})

    peaks_sorted = sorted(peaks, key=lambda x: x["Response"], reverse=True)
    return {
        "freqs": freqs,
        "resp": resp,
        "f_list": f_list,
        "peaks_sorted": peaks_sorted,
        "m_beam_g": m_beam * 1000.0,
    }

# ==========================================
# 3. 2D RIG DRAWING + FREQUENCY RESPONSE PLOT
# ==========================================
def draw_rig_and_response(mat_name, struct_type, L_mm, b_mm, h_mm, tip_mass_g, zeta, f_max_hz, amp_g):
    res = compute_vibration_response(mat_name, struct_type, L_mm, b_mm, h_mm, tip_mass_g, zeta, f_max_hz, amp_g)
    freqs = res["freqs"]
    resp = res["resp"]
    peaks = res["peaks_sorted"]

    plt.style.use("seaborn-v0_8-whitegrid")
    fig, (ax_rig, ax_frf) = plt.subplots(1, 2, figsize=(13.6, 4.6), gridspec_kw={"width_ratios": [1.0, 1.45]})

    # --- LEFT: 2D Test Rig & Dominant Mode Shape ---
    s = np.linspace(0.0, 1.0, 200)
    x_mm = s * L_mm

    if struct_type == "Cantilever Beam":
        mode_shape = 1.0 - np.cos(0.5 * np.pi * s)
        mode_shape = mode_shape / max(np.max(np.abs(mode_shape)), 1e-9)
        ax_rig.add_patch(plt.Rectangle((-0.07 * L_mm, -0.18 * L_mm), 0.07 * L_mm, 0.36 * L_mm,
                                      facecolor="#334155", edgecolor="#0f172a", lw=1.5))
        ax_rig.text(-0.035 * L_mm, -0.24 * L_mm, "Clamp", ha="center", va="top", fontsize=8.5, fontweight="bold")
        ax_rig.scatter([L_mm], [0.08 * L_mm * mode_shape[-1]], color="#f97316", s=65, zorder=6, label="Tip / Sensor")
    elif struct_type == "Simply Supported Beam":
        mode_shape = np.sin(np.pi * s)
        ax_rig.scatter([0, L_mm], [-0.04 * L_mm, -0.04 * L_mm], marker="^", s=90, color="#0f172a", zorder=6)
        ax_rig.text(0.5 * L_mm, -0.14 * L_mm, "Pinned Supports", ha="center", va="top", fontsize=8.5, fontweight="bold")
    else:  # "Small Frame Structure"
        mode_shape = np.sin(np.pi * s)**1.3
        ax_rig.plot([0, 0], [-0.15 * L_mm, 0], color="#0f172a", lw=3.0)
        ax_rig.plot([L_mm, L_mm], [-0.15 * L_mm, 0], color="#0f172a", lw=3.0)
        ax_rig.text(0.5 * L_mm, -0.14 * L_mm, "Frame Base Supports", ha="center", va="top", fontsize=8.5, fontweight="bold")

    amp_vis = 0.10 * L_mm
    ax_rig.plot(x_mm, np.zeros_like(x_mm), color="#94a3b8", ls="--", lw=1.2, label="Static Axis")
    ax_rig.plot(x_mm, mode_shape * amp_vis, color="#2563eb", lw=2.5, label="Dominant Mode Shape")
    ax_rig.annotate("Shaker / Exciter", xy=(0.35 * L_mm, 0.02 * L_mm), xytext=(0.35 * L_mm, -0.24 * L_mm),
                    ha="center", fontsize=8.5, fontweight="bold", color="#dc2626",
                    arrowprops=dict(arrowstyle="->", color="#dc2626", lw=1.4))

    ax_rig.set_title(f"2D Vibration Rig — {struct_type}", fontsize=11, fontweight="bold", pad=10)
    ax_rig.set_xlabel("Structure Span x [mm]", fontsize=9)
    ax_rig.set_ylabel("Deflection [mm] (Schematic)", fontsize=9)
    ax_rig.set_xlim(-0.14 * L_mm, 1.12 * L_mm)
    ax_rig.set_ylim(-0.32 * L_mm, 0.24 * L_mm)
    ax_rig.legend(loc="upper left", fontsize=8, frameon=True, facecolor="white")

    # --- RIGHT: Frequency Response & Largest-Response Frequencies ---
    ax_frf.plot(freqs, resp, color="#2563eb", lw=2.2, label="Vibration Response")

    colors_pk = ["#dc2626", "#f97316", "#eab308"]
    for idx, pk in enumerate(peaks[:3]):
        col = colors_pk[idx % len(colors_pk)]
        ax_frf.scatter([pk["Freq_Hz"]], [pk["Response"]], color=col, s=75, zorder=6, edgecolor="#0f172a", lw=1.0)
        ax_frf.axvline(pk["Freq_Hz"], color=col, ls=":", lw=1.1, alpha=0.8)
        ax_frf.annotate(
            f"#{idx+1} ({pk['Mode']}): {pk['Freq_Hz']:.1f} Hz\nPeak: {pk['Response']:.1f}",
            xy=(pk["Freq_Hz"], pk["Response"]),
            xytext=(8, 6 - idx * 14), textcoords="offset points",
            fontsize=8.2, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.28", facecolor="#fff7ed", edgecolor=col)
        )

    ax_frf.set_title(f"Response vs. Frequency — {mat_name} (ζ = {zeta:.3f})", fontsize=11, fontweight="bold", pad=10)
    ax_frf.set_xlabel("Excitation Frequency [Hz]", fontsize=9.5)
    ax_frf.set_ylabel("Response Amplitude (Arbitrary / g-scale)", fontsize=9.5)
    ax_frf.legend(loc="upper right", fontsize=8.5, frameon=True, facecolor="white")

    plt.tight_layout()
    plt.show()

    # Summary Table & Card
    if peaks:
        tbl_df = pd.DataFrame([
            {"Rank": f"#{i+1}", "Mode": p["Mode"], "Frequency [Hz]": round(p["Freq_Hz"], 2), "Peak Response": round(p["Response"], 2)}
            for i, p in enumerate(peaks[:3])
        ])
        display(tbl_df.style.hide(axis="index"))
        top_str = ", ".join([f"{p['Freq_Hz']:.1f} Hz ({p['Mode']})" for p in peaks[:3]])
    else:
        top_str = "No resonant peak inside current sweep range; increase Max Freq [Hz]."

    display(HTML(f"""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:13px 17px;color:#f8fafc;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;margin-top:8px;
    ">
        <div style="font-size:13px;">
            <b>Largest-Response Frequencies:</b> <span style="color:#7dd3fc;font-weight:700;">{top_str}</span> &nbsp;|&nbsp;
            <b>Beam Mass:</b> {res['m_beam_g']:.1f} g &nbsp;|&nbsp;
            <b>Damping Ratio (ζ):</b> {zeta:.3f}
        </div>
        <div style="font-size:12.5px;color:#cbd5e1;margin-top:4px;line-height:1.5;">
            When excitation frequency matches a natural frequency of the structure, resonance amplifies response peaks;
            higher damping or lower tip mass shifts amplitude and natural frequencies accordingly.
        </div>
    </div>
    """))

# ==========================================
# 4. INTERACTIVE ENGLISH UI
# ==========================================
lbl_style = {"description_width": "145px"}
ctrl_layout = widgets.Layout(width="340px")

w_mat    = widgets.Dropdown(options=list(RIG_MATERIALS.keys()), value="Aluminum 6061", description="Material:", style=lbl_style, layout=ctrl_layout)
w_struct = widgets.Dropdown(options=list(STRUCTURE_TYPES.keys()), value="Cantilever Beam", description="Structure:", style=lbl_style, layout=ctrl_layout)
w_L      = widgets.FloatSlider(value=300.0, min=100.0, max=800.0, step=20.0, description="Length L [mm]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_b      = widgets.FloatSlider(value=25.0,  min=5.0,   max=100.0, step=1.0,  description="Width b [mm]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_h      = widgets.FloatSlider(value=3.0,   min=0.8,   max=15.0,  step=0.2,  description="Thickness h [mm]:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)
w_mtip   = widgets.FloatSlider(value=15.0,  min=0.0,   max=200.0, step=5.0,  description="Tip Mass [g]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_zeta   = widgets.FloatSlider(value=0.015, min=0.002, max=0.100, step=0.002,description="Damping ζ:", readout_format=".3f", style=lbl_style, layout=ctrl_layout)
w_fmax   = widgets.FloatSlider(value=150.0, min=30.0,  max=500.0, step=10.0, description="Max Freq [Hz]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_amp    = widgets.FloatSlider(value=1.0,   min=0.2,   max=5.0,   step=0.2,  description="Base Input [g]:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)

btn_run = widgets.Button(description="Run Vibration Sweep", button_style="primary", icon="bolt", layout=widgets.Layout(width="210px", height="36px"))
out_vib = widgets.Output(layout=widgets.Layout(width="100%"))

def sync_damping_default(change):
    w_zeta.value = RIG_MATERIALS[change["new"]]["zeta_def"]

w_mat.observe(sync_damping_default, names="value")

def refresh_vib(*args):
    with out_vib:
        clear_output(wait=True)
        draw_rig_and_response(
            w_mat.value, w_struct.value, w_L.value, w_b.value, w_h.value,
            w_mtip.value, w_zeta.value, w_fmax.value, w_amp.value
        )

btn_run.on_click(refresh_vib)
for c in [w_mat, w_struct, w_L, w_b, w_h, w_mtip, w_zeta, w_fmax, w_amp]:
    c.observe(refresh_vib, names="value")

header_vib = widgets.HTML("""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:12px 16px;margin-bottom:10px;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
    ">
        <div style="font-size:17px;font-weight:700;color:#f8fafc;">Small Structure Vibration Rig — Resonance &amp; Peak Response</div>
        <div style="font-size:12px;color:#94a3b8;margin-top:2px;">
            Simulate a small structure on a vibration rig and identify frequencies producing the largest response
        </div>
    </div>
""")

controls_vib = widgets.VBox([
    widgets.HBox([
        widgets.VBox([w_mat, w_struct, w_L, w_b, w_h], layout=widgets.Layout(margin="0 28px 0 0", gap="7px")),
        widgets.VBox([w_mtip, w_zeta, w_fmax, w_amp, btn_run], layout=widgets.Layout(gap="7px"))
    ], layout=widgets.Layout(align_items="flex-start", flex_flow="row wrap"))
], layout=widgets.Layout(
    width="100%", padding="14px 16px", border="1px solid #334155",
    border_radius="8px", background_color="#1e293b", margin="0 0 12px 0"
))

display(header_vib, controls_vib, out_vib)
refresh_vib()

HTML(value='\n    <div style="\n        width:100%;box-sizing:border-box;\n        background:linear-gradient(…

Output(layout=Layout(width='100%'))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 1. REPRESENTATIVE STRUCTURE MATERIALS
# ==========================================
RIG_MATERIALS = {
    "Aluminum 6061":  {"E": 68.9e9, "rho": 2700.0, "zeta_def": 0.015},
    "3D-Printed PLA": {"E": 2.8e9,  "rho": 1240.0, "zeta_def": 0.030},
    "Balsa Wood":     {"E": 3.5e9,  "rho": 160.0,  "zeta_def": 0.025},
}

STRUCTURE_TYPES = {
    "Cantilever Beam":       {"beta1": 1.8751, "beta2": 4.6941, "beta3": 7.8548, "eff_mass_ratio": 0.24},
    "Simply Supported Beam": {"beta1": np.pi,  "beta2": 2*np.pi,"beta3": 3*np.pi,"eff_mass_ratio": 0.49},
    "Small Frame Structure": {"beta1": 2.3500, "beta2": 5.1000, "beta3": 8.2000, "eff_mass_ratio": 0.32},
}

# ==========================================
# 2. NATURAL FREQUENCIES & FRF MODEL
# ==========================================
def compute_vibration_response(mat_name, struct_type, L_mm, b_mm, h_mm, tip_mass_g, zeta, f_max_hz, amp_g):
    mat = RIG_MATERIALS[mat_name]
    st = STRUCTURE_TYPES[struct_type]

    L = L_mm * 1e-3
    b = b_mm * 1e-3
    h = h_mm * 1e-3
    m_tip = tip_mass_g * 1e-3

    A = b * h
    I = (b * h**3) / 12.0
    m_beam = mat["rho"] * A * L

    f_list = []
    weights = [1.0, 0.35, 0.15]
    for idx, beta in enumerate([st["beta1"], st["beta2"], st["beta3"]]):
        m_total = max(m_beam + weights[idx] * m_tip, 1e-6)
        fn = (beta**2 / (2.0 * np.pi)) * np.sqrt((mat["E"] * I) / (m_total * (L**3)))
        f_list.append(fn)

    freqs = np.linspace(1.0, max(f_max_hz, 20.0), 1500)
    resp = np.zeros_like(freqs)

    # Include all 3 modes in FRF calculation so tail stiffness is physically accurate
    for k_idx, fn in enumerate(f_list):
        r = freqs / max(fn, 1e-6)
        mag = 1.0 / np.sqrt((1.0 - r**2)**2 + (2.0 * zeta * r)**2)
        mode_gain = 1.0 / ((k_idx + 1)**1.3)
        resp += amp_g * mode_gain * mag

    peaks = []
    for idx, fn in enumerate(f_list):
        if 1.0 <= fn <= f_max_hz:
            total_at_fn = 0.0
            for j_idx, f_other in enumerate(f_list):
                rj = fn / max(f_other, 1e-6)
                total_at_fn += amp_g * (1.0 / ((j_idx + 1)**1.3)) / np.sqrt((1.0 - rj**2)**2 + (2.0 * zeta * rj)**2)
            peaks.append({"Mode": f"Mode {idx+1}", "Freq_Hz": fn, "Response": total_at_fn})

    peaks_sorted = sorted(peaks, key=lambda x: x["Response"], reverse=True)
    return {
        "freqs": freqs,
        "resp": resp,
        "f_list": f_list,
        "peaks_sorted": peaks_sorted,
        "m_beam_g": m_beam * 1000.0,
    }

# ==========================================
# 3. 2D RIG DRAWING + FREQUENCY RESPONSE PLOT
# ==========================================
def draw_rig_and_response(mat_name, struct_type, L_mm, b_mm, h_mm, tip_mass_g, zeta, f_max_hz, amp_g):
    res = compute_vibration_response(mat_name, struct_type, L_mm, b_mm, h_mm, tip_mass_g, zeta, f_max_hz, amp_g)
    freqs = res["freqs"]
    resp = res["resp"]
    peaks = res["peaks_sorted"]
    f1, f2, f3 = res["f_list"]

    plt.style.use("seaborn-v0_8-whitegrid")
    fig, (ax_rig, ax_frf) = plt.subplots(1, 2, figsize=(13.6, 4.6), gridspec_kw={"width_ratios": [1.0, 1.45]})

    # --- LEFT: 2D Test Rig & Dominant Mode Shape ---
    s = np.linspace(0.0, 1.0, 200)
    x_mm = s * L_mm

    if struct_type == "Cantilever Beam":
        mode_shape = 1.0 - np.cos(0.5 * np.pi * s)
        mode_shape = mode_shape / max(np.max(np.abs(mode_shape)), 1e-9)
        ax_rig.add_patch(plt.Rectangle((-0.07 * L_mm, -0.18 * L_mm), 0.07 * L_mm, 0.36 * L_mm,
                                      facecolor="#334155", edgecolor="#0f172a", lw=1.5))
        ax_rig.text(-0.035 * L_mm, -0.24 * L_mm, "Clamp", ha="center", va="top", fontsize=8.5, fontweight="bold")
        ax_rig.scatter([L_mm], [0.10 * L_mm * mode_shape[-1]], color="#f97316", s=65, zorder=6, label="Tip / Sensor")
    elif struct_type == "Simply Supported Beam":
        mode_shape = np.sin(np.pi * s)
        ax_rig.scatter([0, L_mm], [-0.04 * L_mm, -0.04 * L_mm], marker="^", s=90, color="#0f172a", zorder=6)
        ax_rig.text(0.5 * L_mm, -0.14 * L_mm, "Pinned Supports", ha="center", va="top", fontsize=8.5, fontweight="bold")
    else:
        mode_shape = np.sin(np.pi * s)**1.3
        ax_rig.plot([0, 0], [-0.15 * L_mm, 0], color="#0f172a", lw=3.0)
        ax_rig.plot([L_mm, L_mm], [-0.15 * L_mm, 0], color="#0f172a", lw=3.0)
        ax_rig.text(0.5 * L_mm, -0.14 * L_mm, "Frame Base Supports", ha="center", va="top", fontsize=8.5, fontweight="bold")

    amp_vis = 0.10 * L_mm
    ax_rig.plot(x_mm, np.zeros_like(x_mm), color="#94a3b8", ls="--", lw=1.2, label="Static Axis")
    ax_rig.plot(x_mm, mode_shape * amp_vis, color="#2563eb", lw=2.5, label="Dominant Mode Shape")
    ax_rig.annotate("Shaker / Exciter", xy=(0.35 * L_mm, 0.02 * L_mm), xytext=(0.35 * L_mm, -0.24 * L_mm),
                    ha="center", fontsize=8.5, fontweight="bold", color="#dc2626",
                    arrowprops=dict(arrowstyle="->", color="#dc2626", lw=1.4))

    ax_rig.set_title(f"2D Vibration Rig — {struct_type}", fontsize=11, fontweight="bold", pad=10)
    ax_rig.set_xlabel("Structure Span x [mm]", fontsize=9)
    ax_rig.set_ylabel("Deflection [mm] (Schematic)", fontsize=9)
    ax_rig.set_xlim(-0.14 * L_mm, 1.12 * L_mm)
    ax_rig.set_ylim(-0.32 * L_mm, 0.24 * L_mm)
    ax_rig.legend(loc="upper left", fontsize=8, frameon=True, facecolor="white")

    # --- RIGHT: Frequency Response & Largest-Response Frequencies ---
    ax_frf.plot(freqs, resp, color="#2563eb", lw=2.2, label="Vibration Response")

    colors_pk = ["#dc2626", "#f97316", "#eab308"]
    for idx, pk in enumerate(peaks[:3]):
        col = colors_pk[idx % len(colors_pk)]
        ax_frf.scatter([pk["Freq_Hz"]], [pk["Response"]], color=col, s=75, zorder=6, edgecolor="#0f172a", lw=1.0)
        ax_frf.axvline(pk["Freq_Hz"], color=col, ls=":", lw=1.1, alpha=0.8)
        ax_frf.annotate(
            f"#{idx+1} ({pk['Mode']}): {pk['Freq_Hz']:.1f} Hz\nPeak: {pk['Response']:.1f}",
            xy=(pk["Freq_Hz"], pk["Response"]),
            xytext=(8, 4 - idx * 14), textcoords="offset points",
            fontsize=8.2, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.28", facecolor="#fff7ed", edgecolor=col)
        )

    ax_frf.set_title(f"Response vs. Frequency — {mat_name} (ζ = {zeta:.3f})", fontsize=11, fontweight="bold", pad=10)
    ax_frf.set_xlabel("Excitation Frequency [Hz]", fontsize=9.5)
    ax_frf.set_ylabel("Response Amplitude (g-scale)", fontsize=9.5)
    ax_frf.legend(loc="upper right", fontsize=8.5, frameon=True, facecolor="white")

    plt.tight_layout()
    plt.show()

    if peaks:
        tbl_df = pd.DataFrame([
            {"Rank": f"#{i+1}", "Mode": p["Mode"], "Frequency [Hz]": round(p["Freq_Hz"], 2), "Peak Response": round(p["Response"], 2)}
            for i, p in enumerate(peaks[:3])
        ])
        display(tbl_df.style.hide(axis="index"))
        top_str = ", ".join([f"{p['Freq_Hz']:.1f} Hz ({p['Mode']})" for p in peaks[:3]])
    else:
        top_str = f"Above {f_max_hz:.0f} Hz sweep (Est. f1 ≈ {f1:.1f} Hz, f2 ≈ {f2:.1f} Hz) — increase Max Freq [Hz]"

    display(HTML(f"""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:13px 17px;color:#f8fafc;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;margin-top:8px;
    ">
        <div style="font-size:13px;">
            <b>Largest-Response Frequencies:</b> <span style="color:#7dd3fc;font-weight:700;">{top_str}</span> &nbsp;|&nbsp;
            <b>Estimated f1 / f2 / f3:</b> {f1:.1f} / {f2:.1f} / {f3:.1f} Hz &nbsp;|&nbsp;
            <b>Beam Mass:</b> {res['m_beam_g']:.1f} g
        </div>
        <div style="font-size:12.5px;color:#cbd5e1;margin-top:4px;line-height:1.5;">
            Stiffer or shorter structures have higher natural frequencies; if no peak appears inside the sweep range,
            increase <b>Max Freq [Hz]</b> or use a longer/thinner structure to bring resonance into the sweep band.
        </div>
    </div>
    """))

# ==========================================
# 4. INTERACTIVE ENGLISH UI
# ==========================================
lbl_style = {"description_width": "145px"}
ctrl_layout = widgets.Layout(width="340px")

w_mat    = widgets.Dropdown(options=list(RIG_MATERIALS.keys()), value="Aluminum 6061", description="Material:", style=lbl_style, layout=ctrl_layout)
w_struct = widgets.Dropdown(options=list(STRUCTURE_TYPES.keys()), value="Cantilever Beam", description="Structure:", style=lbl_style, layout=ctrl_layout)
w_L      = widgets.FloatSlider(value=350.0, min=100.0, max=800.0, step=20.0, description="Length L [mm]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_b      = widgets.FloatSlider(value=25.0,  min=5.0,   max=100.0, step=1.0,  description="Width b [mm]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_h      = widgets.FloatSlider(value=2.0,   min=0.5,   max=15.0,  step=0.2,  description="Thickness h [mm]:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)
w_mtip   = widgets.FloatSlider(value=25.0,  min=0.0,   max=200.0, step=5.0,  description="Tip Mass [g]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_zeta   = widgets.FloatSlider(value=0.015, min=0.002, max=0.100, step=0.002,description="Damping ζ:", readout_format=".3f", style=lbl_style, layout=ctrl_layout)
w_fmax   = widgets.FloatSlider(value=400.0, min=50.0,  max=2500.0,step=25.0, description="Max Freq [Hz]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_amp    = widgets.FloatSlider(value=1.0,   min=0.2,   max=5.0,   step=0.2,  description="Base Input [g]:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)

btn_run = widgets.Button(description="Run Vibration Sweep", button_style="primary", icon="bolt", layout=widgets.Layout(width="210px", height="36px"))
out_vib = widgets.Output(layout=widgets.Layout(width="100%"))

def sync_damping_default(change):
    w_zeta.value = RIG_MATERIALS[change["new"]]["zeta_def"]

w_mat.observe(sync_damping_default, names="value")

def refresh_vib(*args):
    with out_vib:
        clear_output(wait=True)
        draw_rig_and_response(
            w_mat.value, w_struct.value, w_L.value, w_b.value, w_h.value,
            w_mtip.value, w_zeta.value, w_fmax.value, w_amp.value
        )

btn_run.on_click(refresh_vib)
for c in [w_mat, w_struct, w_L, w_b, w_h, w_mtip, w_zeta, w_fmax, w_amp]:
    c.observe(refresh_vib, names="value")

header_vib = widgets.HTML("""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:12px 16px;margin-bottom:10px;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
    ">
        <div style="font-size:17px;font-weight:700;color:#f8fafc;">Small Structure Vibration Rig — Resonance &amp; Peak Response</div>
        <div style="font-size:12px;color:#94a3b8;margin-top:2px;">
            Simulate a small structure on a vibration rig and identify frequencies producing the largest response
        </div>
    </div>
""")

controls_vib = widgets.VBox([
    widgets.HBox([
        widgets.VBox([w_mat, w_struct, w_L, w_b, w_h], layout=widgets.Layout(margin="0 28px 0 0", gap="7px")),
        widgets.VBox([w_mtip, w_zeta, w_fmax, w_amp, btn_run], layout=widgets.Layout(gap="7px"))
    ], layout=widgets.Layout(align_items="flex-start", flex_flow="row wrap"))
], layout=widgets.Layout(
    width="100%", padding="14px 16px", border="1px solid #334155",
    border_radius="8px", background_color="#1e293b", margin="0 0 12px 0"
))

display(header_vib, controls_vib, out_vib)
refresh_vib()

HTML(value='\n    <div style="\n        width:100%;box-sizing:border-box;\n        background:linear-gradient(…

Output(layout=Layout(width='100%'))